# Procurement Item Categorisation

This notebook demonstrates the reusable package with fully synthetic data.
It combines product, manufacturer, catalogue and vendor text; trains a
TF-IDF/LinearSVC classifier; evaluates long-tail performance; and routes
low-margin predictions to human review.

Commercial data, prediction exports and trained model artefacts must remain
outside the public repository.


## Setup

From the repository root, install the package before running the notebook:

```bash
pip install -e ".[dev]"
```


In [ ]:
import os
from pathlib import Path

import pandas as pd

from item_categorisation.features import (
    DEFAULT_FEATURE_COLUMNS,
    build_text_feature,
    validate_columns,
)
from item_categorisation.model import (
    build_classifier,
    evaluate_classifier,
    long_tail_train_test_split,
    predict_with_review,
    select_review_threshold,
)
from item_categorisation.synthetic import write_synthetic_data


## Configuration


In [ ]:
TRAINING_DATA_PATH = Path(
    os.getenv(
        "ITEM_CAT_TRAINING_DATA",
        "examples/data/training_synthetic.csv",
    )
)
PREDICTION_DATA_PATH = Path(
    os.getenv(
        "ITEM_CAT_PREDICTION_DATA",
        "examples/data/prediction_synthetic.csv",
    )
)
OUTPUT_PATH = Path(
    os.getenv(
        "ITEM_CAT_OUTPUT",
        "outputs/synthetic_proposed_categories.csv",
    )
)

TARGET_COLUMN = "Category"
FEATURE_COLUMNS = DEFAULT_FEATURE_COLUMNS
TEST_SIZE = 0.20
RANDOM_STATE = 42


## Create the synthetic example when required


In [ ]:
if (
    not TRAINING_DATA_PATH.exists()
    or not PREDICTION_DATA_PATH.exists()
):
    default_example_dir = Path("examples/data")
    expected_paths = {
        default_example_dir / "training_synthetic.csv",
        default_example_dir / "prediction_synthetic.csv",
    }
    requested_paths = {
        TRAINING_DATA_PATH,
        PREDICTION_DATA_PATH,
    }
    if requested_paths != expected_paths:
        raise FileNotFoundError(
            "A configured private data path does not exist."
        )
    write_synthetic_data(
        default_example_dir,
        random_state=RANDOM_STATE,
    )


## Load and validate labelled data


In [ ]:
training_data = pd.read_csv(
    TRAINING_DATA_PATH,
    low_memory=False,
)
validate_columns(
    training_data,
    [*FEATURE_COLUMNS, TARGET_COLUMN],
    "Training data",
)
training_data = training_data.dropna(
    subset=[TARGET_COLUMN]
).copy()
training_data["ModelText"] = build_text_feature(
    training_data,
    FEATURE_COLUMNS,
)
training_data = training_data.loc[
    training_data["ModelText"].ne("")
].reset_index(drop=True)

category_counts = training_data[
    TARGET_COLUMN
].value_counts()
pd.Series(
    {
        "Rows": len(training_data),
        "Categories": category_counts.size,
        "Singleton categories": int(
            (category_counts == 1).sum()
        ),
        "Categories below five rows": int(
            (category_counts < 5).sum()
        ),
        "Median rows per category": float(
            category_counts.median()
        ),
    },
    name="Value",
).to_frame()


## Long-tail holdout and model training


In [ ]:
train_rows, test_rows = long_tail_train_test_split(
    training_data,
    target_column=TARGET_COLUMN,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

classifier = build_classifier(
    random_state=RANDOM_STATE
)
classifier.fit(
    train_rows["ModelText"],
    train_rows[TARGET_COLUMN],
)

pd.Series(
    {
        "Training rows": len(train_rows),
        "Holdout rows": len(test_rows),
    },
    name="Value",
).to_frame()


## Evaluation


In [ ]:
metrics, per_category, y_pred, margins = (
    evaluate_classifier(
        classifier,
        test_rows["ModelText"],
        test_rows[TARGET_COLUMN],
    )
)

pd.Series(metrics, name="Value").to_frame()


In [ ]:
per_category.head(20).round(
    {
        "precision": 3,
        "recall": 3,
        "f1-score": 3,
        "support": 0,
    }
)


Macro F1 gives each category equal weight and is therefore particularly useful
for the long tail. Top-k accuracy measures whether the correct category appears
among the strongest alternatives.


## Derive a human-review policy


In [ ]:
review_policy = select_review_threshold(
    test_rows[TARGET_COLUMN],
    y_pred,
    margins,
    target_auto_accuracy=0.90,
    minimum_auto_coverage=0.20,
)

pd.Series(
    review_policy,
    name="Value",
).to_frame()


The selected threshold is based on holdout performance. It is not a universal
confidence score and must be monitored after deployment.


## Predict new items without changing row count


In [ ]:
prediction_input = pd.read_csv(
    PREDICTION_DATA_PATH,
    low_memory=False,
)
validate_columns(
    prediction_input,
    FEATURE_COLUMNS,
    "Prediction data",
)
prediction_text = build_text_feature(
    prediction_input,
    FEATURE_COLUMNS,
)
if prediction_text.eq("").any():
    raise ValueError(
        "Prediction data contains rows with no usable text."
    )

predictions = predict_with_review(
    classifier,
    prediction_text,
    review_threshold=review_policy["threshold"],
    alternative_count=2,
)

prediction_output = (
    prediction_input.reset_index(drop=True)
    .join(predictions)
)
assert len(prediction_output) == len(prediction_input)

prediction_output.head()


## Export the synthetic demonstration


In [ ]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)
prediction_output.to_csv(
    OUTPUT_PATH,
    index=False,
)

pd.Series(
    {
        "Input rows": len(prediction_input),
        "Output rows": len(prediction_output),
        "Review rows": int(
            prediction_output[
                "ReviewRecommended"
            ].sum()
        ),
        "Output path": str(OUTPUT_PATH),
    },
    name="Value",
).to_frame()


## Production considerations

- Use grouped or time-based validation when stable item identifiers exist.
- Review high-value items even when the model margin is high.
- Version the model, taxonomy and source data together.
- Monitor accepted accuracy, review coverage and category drift.
- Feed only approved reviewer corrections into future training.
- Keep source records, exports and model bundles in access-controlled storage.
